In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration
Connected to future database: DB_FUTURE


In [7]:
import pandas as pd
import pickle

# =========================================================
# 1. EXTRACT DATA DARI DB LAMA
# =========================================================
print("📥 Mengambil data jadwal dari database lama...")
cursor_old.execute("SELECT * FROM jadwal")
data_jadwal_old = cursor_old.fetchall()
df_raw_jadwal = pd.DataFrame(data_jadwal_old)
print(f"Total data asli di DB lama: {len(df_raw_jadwal)} baris")

# Extract tabel jadwal
df_jadwal_lama = pd.read_sql("SELECT * FROM jadwal", db_old)

# =========================================================
# 2. TRANSFORMASI TABEL: jadwal (seperti kode Anda)
# =========================================================
print("⚡ Melakukan transformasi tabel 'jadwal'...")

# A. Pembersihan metode belajar
def clean_mode_belajar(val):
    if pd.isna(val) or not str(val).strip():
        return 'Offline'
    s = str(val).strip().capitalize()
    if s in ('Online', 'Offline', 'Hybrid'):
        return s
    return 'Offline'

# B. Pembersihan status arsip
def clean_status_arsip(val):
    try:
        if pd.isna(val):
            return 0
        return int(float(val))
    except:
        return 0

# C. Saring (filter out) jadwal percobaan
df_jadwal_clean = df_raw_jadwal[~df_raw_jadwal['idperiode'].isin(['P00094', 'P00104'])].copy()
skipped_count = len(df_raw_jadwal) - len(df_jadwal_clean)
print(f"ℹ️ Menyaring {skipped_count} data jadwal percobaan. Sisa data: {len(df_jadwal_clean)} baris")

# Siapkan dataframe untuk insert (tanpa id)
df_jadwal_insert = pd.DataFrame({
    'id_kursus': df_jadwal_clean['idpendkursus'],
    'id_periode': df_jadwal_clean['idperiode'],
    'id_level': df_jadwal_clean['idlevel'],
    'id_sesi': df_jadwal_clean['idsesi'],
    'metode_belajar_jadwal': df_jadwal_clean['mode_belajar'].apply(clean_mode_belajar),
    'nama_rombel': df_jadwal_clean['groupwa'].fillna('').str.strip(),
    'status_arsip': df_jadwal_clean['status_archive'].apply(clean_status_arsip),
    'tempat': df_jadwal_clean['tempat'].fillna('Ruang Kelas').replace('', 'Ruang Kelas').str.strip()
})

# Simpan urutan old_id (sesuai urutan df_jadwal_insert)
old_id_list = df_jadwal_clean['idjadwal'].tolist()

# =========================================================
# BUAT ID BARU (SIMULASI AUTO INCREMENT) & MAPPING
# =========================================================
print("🔢 Membuat ID baru (auto increment simulasi)...")
# Tambahkan kolom 'id_jadwal' dengan angka urut mulai dari 1
df_jadwal_insert.insert(0, 'id_jadwal', range(1, len(df_jadwal_insert) + 1))

# Buat mapping old -> new
mapping_id_jadwal = dict(zip(old_id_list, df_jadwal_insert['id_jadwal'].tolist()))
print(f"✅ Mapping ID jadwal selesai. Jumlah: {len(mapping_id_jadwal)}")

# Simpan mapping ke file pickle (opsional)
with open('mapping_id_jadwal.pkl', 'wb') as f:
    pickle.dump(mapping_id_jadwal, f)
print("💾 Mapping disimpan ke 'mapping_id_jadwal.pkl'")

# =========================================================
# 3. TRANSFORMASI TABEL BARU: jadwal_hari (menggunakan mapping)
# =========================================================
print("⚡ Memisahkan kolom 'hari' ke tabel 'jadwal_hari'...")

hari_rows = []
for idx, row in df_jadwal_clean.iterrows():
    old_id = row['idjadwal']
    hari_string = row['hari']
    if pd.isna(hari_string) or not str(hari_string).strip():
        continue
    for hari in [h.strip() for h in hari_string.split(',') if h.strip()]:
        hari_rows.append({
            'id_jadwal': old_id,   # masih old id
            'nama_hari': hari
        })

df_jadwal_hari = pd.DataFrame(hari_rows)

# Ganti id_jadwal dengan ID baru menggunakan mapping
df_jadwal_hari['id_jadwal'] = df_jadwal_hari['id_jadwal'].map(mapping_id_jadwal)
# Hapus baris yang tidak punya mapping (jika ada)
df_jadwal_hari = df_jadwal_hari.dropna(subset=['id_jadwal'])
df_jadwal_hari['id_jadwal'] = df_jadwal_hari['id_jadwal'].astype(int)

print(f"✓ Tabel 'jadwal_hari' siap. Shape: {df_jadwal_hari.shape}")

# =========================================================
# 4. SIMPAN SEMUA DATA UNTUK TAHAP SELANJUTNYA
# =========================================================
fase_4_afrida = {
    'jadwal': df_jadwal_insert,          # sudah punya id baru
    'jadwal_old_ids': old_id_list,       # urutan lama (untuk referensi)
    'jadwal_hari': df_jadwal_hari,       # sudah pakai id baru
    'mapping_id_jadwal': mapping_id_jadwal,
    # nanti tambahkan 'jadwal_detail', 'jadwal_pengajar', 'jadwal_siswa', dll.
}

# Simpan ke pickle
with open('fase_4_afrida_v2.pkl', 'wb') as f:
    pickle.dump(fase_4_afrida, f)
print("💾 Semua data disimpan ke 'fase_4_afrida.pkl'")

📥 Mengambil data jadwal dari database lama...
Total data asli di DB lama: 551 baris
⚡ Melakukan transformasi tabel 'jadwal'...
ℹ️ Menyaring 2 data jadwal percobaan. Sisa data: 549 baris
🔢 Membuat ID baru (auto increment simulasi)...
✅ Mapping ID jadwal selesai. Jumlah: 549
💾 Mapping disimpan ke 'mapping_id_jadwal.pkl'
⚡ Memisahkan kolom 'hari' ke tabel 'jadwal_hari'...
✓ Tabel 'jadwal_hari' siap. Shape: (975, 2)
💾 Semua data disimpan ke 'fase_4_afrida.pkl'


In [8]:
display(df_jadwal_insert.head())

,id_jadwal,id_kursus,id_periode,id_level,id_sesi,metode_belajar_jadwal,nama_rombel,status_arsip,tempat
0,1,K00001,P00006,L00017,S00002,Online,01 GOGO 3B SR2 (ERICA),1,Ruang Kelas 4
1,2,K00001,P00006,L00024,S00002,Offline,02 SO 1C SR2 (QORIN),1,Ruang Kelas 5
2,3,K00001,P00006,L00023,S00001,Offline,03 SO 1B SR1 (TATIK),1,Ruang Kelas 1
3,4,K00001,P00006,L00025,S00003,Offline,04 SO 2A SR3 (TATIK),1,Ruang Kelas 1
4,5,K00001,P00006,L00014,S00003,Offline,05 GOGO 1B SelK3 (ERICA),1,Ruang Kelas 4


In [46]:
# =========================================================
# JADWAL_DETAIL - PEMBUANGAN ORPHAN DENGAN INNER JOIN
# =========================================================
print("⚡ Memproses jadwal_detail dengan pembersihan total orphan via inner join...")

# 1. Ambil data mentah
df_detil_lama = pd.read_sql("SELECT * FROM jadwal_detil", db_old)
total_awal = len(df_detil_lama)
print(f"Total detail di DB lama: {total_awal}")

# 2. Simpan old_detail_id dan old_id_jadwal
df_detil_lama['old_detail_id'] = df_detil_lama['idjadwaldetil'].astype(str)
df_detil_lama['old_id_jadwal'] = df_detil_lama['idjadwal'].astype(str)

# 3. Buat dataframe dengan kolom yang diperlukan
df_temp = pd.DataFrame({
    'old_detail_id': df_detil_lama['old_detail_id'],
    'old_id_jadwal': df_detil_lama['old_id_jadwal'],
    'judul': df_detil_lama['title'].fillna('').astype(str),
    'deskripsi': df_detil_lama['description'].fillna('').astype(str),
    'url_jadwal_detail': df_detil_lama['url'].fillna('').astype(str),
    'label_warna': df_detil_lama['color'].fillna('').astype(str),
    'penanda_mulai': pd.to_datetime(df_detil_lama['start'], errors='coerce').dt.date,
    'penanda_selesai': pd.to_datetime(df_detil_lama['end'], errors='coerce').dt.date,
})

# 4. Terapkan mapping untuk mendapatkan id_jadwal baru
df_temp['id_jadwal'] = df_temp['old_id_jadwal'].map(mapping_id_jadwal)

# 5. Hapus baris yang tidak terpetakan (orphan)
before_drop = len(df_temp)
df_temp = df_temp.dropna(subset=['id_jadwal'])
after_drop = len(df_temp)
print(f"Orphan yang dibuang (tidak ada di mapping): {before_drop - after_drop}")

# 6. INNER JOIN dengan df_jadwal_insert untuk memastikan hanya yang valid
# Ambil hanya kolom 'id' dari jadwal_insert
df_jadwal_pk = df_jadwal_insert[['id_jadwal_detail']].copy()
df_temp = df_temp.merge(df_jadwal_pk, left_on='id_jadwal', right_on='id_jadwal_detail', how='inner')
after_join = len(df_temp)
print(f"Barang yang dibuang saat inner join (tidak cocok dengan PK jadwal): {after_drop - after_join}")
    
# Hapus kolom 'id_jadwal_detail' hasil merge (karena kita pakai id_jadwal sebagai FK)
df_temp = df_temp.drop(columns=['id_jadwal_detail'])

# 7. Tambahkan kolom default
df_temp['id_mitra'] = None
df_temp['id_sesi_override'] = None
df_temp['status_detail'] = 'scheduled'
df_temp['source_type'] = 'generated'
df_temp['original_jadwal_detail_id'] = None
df_temp['has_operational_data'] = 0
df_temp['created_at'] = pd.Timestamp.now()
df_temp['updated_at'] = pd.Timestamp.now()
df_temp['last_generated_at'] = df_temp['created_at']

# 8. Cleaning deskripsi & url
df_temp['deskripsi'] = df_temp['deskripsi'].fillna('').replace('', 'Tidak ada deskripsi').str.strip()
df_temp['url_jadwal_detail'] = df_temp['url_jadwal_detail'].fillna('').replace('-', 'Link belum tersedia').str.strip()

# 9. Urutkan ulang kolom (opsional)
df_jadwal_detail_insert = df_temp[[
    'old_detail_id',
    'id_jadwal',
    'judul',
    'deskripsi',
    'url_jadwal_detail',
    'label_warna',
    'penanda_mulai',
    'penanda_selesai',
    'id_mitra',
    'id_sesi_override',
    'status_detail',
    'source_type',
    'original_jadwal_detail_id',
    'has_operational_data',
    'last_generated_at',
    'created_at',
    'updated_at'
]].copy()

# 10. Tambahkan id baru (auto increment) dan buat mapping detail
df_jadwal_detail_insert.insert(0, 'id_jadwal_detail', range(1, len(df_jadwal_detail_insert) + 1))
mapping_id_jadwal_detail = dict(zip(df_jadwal_detail_insert['old_detail_id'], df_jadwal_detail_insert['id_jadwal_detail']))

# Hapus kolom old_detail_id
df_jadwal_detail_insert = df_jadwal_detail_insert.drop(columns=['old_detail_id'])

# 11. Simpan ke fase_4_afrida
fase_4_afrida['jadwal_detail'] = df_jadwal_detail_insert
fase_4_afrida['jadwal_detail_old_ids'] = list(mapping_id_jadwal_detail.keys())
fase_4_afrida['mapping_id_jadwal_detail'] = mapping_id_jadwal_detail

with open('fase_4_afrida_v2.pkl', 'wb') as f:
    pickle.dump(fase_4_afrida, f)

print(f"✅ jadwal_detail selesai. Jumlah akhir: {len(df_jadwal_detail_insert)}")
print(f"✅ Mapping detail siap dengan {len(mapping_id_jadwal_detail)} entri")

⚡ Memproses jadwal_detail dengan pembersihan total orphan via inner join...
Total detail di DB lama: 17267
Orphan yang dibuang (tidak ada di mapping): 10


KeyError: "None of [Index(['id_jadwal_detail'], dtype='object')] are in the [columns]"

In [44]:
# Verifikasi FK
pk_jadwal = set(df_jadwal_insert['id_jadwal'])
fk_detail = set(df_jadwal_detail_insert['id_jadwal_detail'])
invalid = fk_detail - pk_jadwal

print(f"Total detail: {len(df_jadwal_detail_insert)}")
print(f"Total jadwal: {len(df_jadwal_insert)}")
print(f"Jumlah FK invalid: {len(invalid)}")
if len(invalid) == 0:
    print("✅ Semua FK terhubung dengan benar!")
else:
    print("❌ Masih ada FK invalid. Periksa mapping.")

Total detail: 17257
Total jadwal: 549
Jumlah FK invalid: 16708
❌ Masih ada FK invalid. Periksa mapping.


In [39]:
# =========================================================
# ANALISIS DATA ORPHAN (jadwal_detil tanpa parent)
# =========================================================
print("="*70)
print("📊 ANALISIS DATA ORPHAN PADA jadwal_detil")
print("="*70)

# Ambil data detail mentah dari database lama
df_detil_all = pd.read_sql("SELECT * FROM jadwal_detil", db_old)
print(f"Total data detail di DB lama: {len(df_detil_all)} baris")

# ID jadwal yang valid (ada di mapping)
valid_jadwal_ids = set(mapping_id_jadwal.keys())

# Pisahkan orphan
df_orphan = df_detil_all[~df_detil_all['idjadwal'].astype(str).isin(valid_jadwal_ids)].copy()
print(f"\nData orphan (tanpa parent): {len(df_orphan)} baris ({len(df_orphan)/len(df_detil_all)*100:.2f}%)")

# =========================================================
# 1. Contoh data orphan
# =========================================================
print("\n📋 Contoh 10 data orphan:")
display(df_orphan[['idjadwal', 'title', 'start', 'end', 'idjadwaldetil']].head(10))

# =========================================================
# 2. Distribusi orphan berdasarkan idjadwal
# =========================================================
orphan_by_id = df_orphan.groupby('idjadwal').size().sort_values(ascending=False)
print(f"\n📊 Jumlah orphan per idjadwal (total unique: {len(orphan_by_id)})")
print("Top 10 idjadwal dengan orphan terbanyak:")
display(orphan_by_id.head(10))

# =========================================================
# 3. Cek apakah idjadwal orphan termasuk dalam filter sandbox
# =========================================================
# Ambil data jadwal asli untuk melihat idperiode
df_jadwal_all = pd.read_sql("SELECT idjadwal, idperiode, mode_belajar, groupwa FROM jadwal", db_old)
orphan_ids = df_orphan['idjadwal'].unique()

# Filter jadwal yang menjadi orphan
df_orphan_jadwal_info = df_jadwal_all[df_jadwal_all['idjadwal'].isin(orphan_ids)].drop_duplicates()
print(f"\n🏷️ Informasi jadwal yang menjadi parent orphan (total: {len(df_orphan_jadwal_info)})")
print("Distribusi periode:")
display(df_orphan_jadwal_info['idperiode'].value_counts())

# =========================================================
# 4. Lihat apakah orphan terkait dengan periode yang sudah difilter
# =========================================================
filtered_periode = ['P00094', 'P00104']
orphan_in_filtered = df_orphan_jadwal_info[df_orphan_jadwal_info['idperiode'].isin(filtered_periode)]
print(f"\n🚫 Orphan yang berasal dari periode sandbox ({filtered_periode}): {len(orphan_in_filtered)} idjadwal")
print("Contoh:")
display(orphan_in_filtered[['idjadwal', 'idperiode', 'mode_belajar']].head())

orphan_not_filtered = df_orphan_jadwal_info[~df_orphan_jadwal_info['idperiode'].isin(filtered_periode)]
print(f"\n⚠️ Orphan yang BUKAN dari periode sandbox: {len(orphan_not_filtered)} idjadwal")
print("Contoh:")
display(orphan_not_filtered[['idjadwal', 'idperiode', 'mode_belajar']].head())

# =========================================================
# 5. Tampilkan beberapa detail dari orphan yang bukan sandbox
# =========================================================
if len(orphan_not_filtered) > 0:
    ids_not_sandbox = set(orphan_not_filtered['idjadwal'])
    df_orphan_detail_not_sandbox = df_orphan[df_orphan['idjadwal'].isin(ids_not_sandbox)]
    print(f"\n📝 Contoh detail orphan (bukan sandbox):")
    display(df_orphan_detail_not_sandbox[['idjadwal', 'title', 'start', 'end']].head(10))

📊 ANALISIS DATA ORPHAN PADA jadwal_detil
Total data detail di DB lama: 17267 baris

Data orphan (tanpa parent): 10 baris (0.06%)

📋 Contoh 10 data orphan:


,idjadwal,title,start,end,idjadwaldetil
16741,J000000572,coba rapot,2025-01-07,2025-01-07,D0000000000000025145
16742,J000000572,coba rapot,2025-01-14,2025-01-14,D0000000000000025146
16743,J000000572,coba rapot,2025-01-21,2025-01-21,D0000000000000025147
16744,J000000572,coba rapot,2025-02-04,2025-02-04,D0000000000000025148
16745,J000000572,coba rapot,2025-02-11,2025-02-11,D0000000000000025149
16746,J000000573,Coba raport 2,2026-03-02,2026-03-02,D0000000000000025150
16747,J000000573,Coba raport 2,2026-03-09,2026-03-09,D0000000000000025151
16748,J000000573,Coba raport 2,2026-03-16,2026-03-16,D0000000000000025152
16749,J000000573,Coba raport 2,2026-03-30,2026-03-30,D0000000000000025153
16750,J000000573,Coba raport 2,2026-04-06,2026-04-06,D0000000000000025154



📊 Jumlah orphan per idjadwal (total unique: 2)
Top 10 idjadwal dengan orphan terbanyak:


idjadwal
J000000572    5
J000000573    5
dtype: int64


🏷️ Informasi jadwal yang menjadi parent orphan (total: 2)
Distribusi periode:


idperiode
P00094    1
P00104    1
Name: count, dtype: int64


🚫 Orphan yang berasal dari periode sandbox (['P00094', 'P00104']): 2 idjadwal
Contoh:


,idjadwal,idperiode,mode_belajar
529,J000000572,P00094,Offline
530,J000000573,P00104,Online



⚠️ Orphan yang BUKAN dari periode sandbox: 0 idjadwal
Contoh:


,idjadwal,idperiode,mode_belajar


In [29]:
# =========================================================
# VERIFIKASI FOREIGN KEY: jadwal_detail -> jadwal
# =========================================================
print("="*70)
print("🔍 MEMERIKSA RELASI FK: df_jadwal_detail_insert -> df_jadwal_insert")
print("="*70)

# Ambil set ID jadwal baru (primary key)
pk_jadwal = set(df_jadwal_insert['id_jadwal'])

# Set ID jadwal di tabel detail (foreign key)
fk_jadwal_detail = set(df_jadwal_detail_insert['id_jadwal'])

# 1. Cek NULL pada kolom id_jadwal
null_count = df_jadwal_detail_insert['id_jadwal'].isna().sum()
print(f"1. Jumlah nilai NULL pada kolom 'id_jadwal' di detail: {null_count}")
if null_count > 0:
    print("   ⚠️ Ada NULL! Perbaiki mapping.")
else:
    print("   ✅ Tidak ada NULL.")

# 2. Cek apakah semua FK ada di PK
invalid_fk = fk_jadwal_detail - pk_jadwal
print(f"2. Jumlah id_jadwal di detail yang TIDAK ADA di tabel jadwal: {len(invalid_fk)}")
if len(invalid_fk) > 0:
    print(f"   ❌ ID tidak valid (contoh 5): {list(invalid_fk)[:5]}")
else:
    print("   ✅ Semua FK valid.")

# 3. Statistik jumlah data
total_detail = len(df_jadwal_detail_insert)
total_jadwal = len(pk_jadwal)
print(f"3. Total data jadwal_detail: {total_detail}")
print(f"   Total data jadwal: {total_jadwal}")
print(f"   Jumlah unik id_jadwal di detail: {len(fk_jadwal_detail)}")

# 4. Cek orphan (detail tanpa parent) – seharusnya 0
orphan = df_jadwal_detail_insert[~df_jadwal_detail_insert['id_jadwal'].isin(pk_jadwal)]
print(f"4. Jumlah data detail orphan (tanpa parent): {len(orphan)}")
if len(orphan) > 0:
    print("   ❌ Ada orphan! Periksa data berikut:")
    display(orphan[['id_jadwal_detail', 'id_jadwal', 'judul']].head(10))

# 5. Tampilkan contoh join (beberapa baris) untuk inspeksi visual
print("\n5. Contoh data detail dengan parent-nya (5 baris pertama):")
# Gabungkan dengan jadwal untuk melihat nama rombel atau info lain sebagai sampel
sample_join = df_jadwal_detail_insert[['id_jadwal_detail', 'id_jadwal', 'judul']].head(5).merge(
    df_jadwal_insert[['id_jadwal', 'nama_rombel', 'metode_belajar_jadwal']],
    left_on='id_jadwal_detail', right_on='id_jadwal', how='left'
)
display(sample_join)

print("="*70)
if null_count == 0 and len(invalid_fk) == 0 and len(orphan) == 0:
    print("✅ VERIFIKASI LULUS: Semua FK terhubung dengan benar.")
else:
    print("❌ VERIFIKASI GAGAL: Ada masalah pada relasi FK. Perbaiki sebelum lanjut.")

🔍 MEMERIKSA RELASI FK: df_jadwal_detail_insert -> df_jadwal_insert
1. Jumlah nilai NULL pada kolom 'id_jadwal' di detail: 0
   ✅ Tidak ada NULL.
2. Jumlah id_jadwal di detail yang TIDAK ADA di tabel jadwal: 0
   ✅ Semua FK valid.
3. Total data jadwal_detail: 17257
   Total data jadwal: 549
   Jumlah unik id_jadwal di detail: 549
4. Jumlah data detail orphan (tanpa parent): 0

5. Contoh data detail dengan parent-nya (5 baris pertama):


,id_jadwal_detail,id_jadwal_x,judul,id_jadwal_y,nama_rombel,metode_belajar_jadwal
0,1,8,08 SO 2A SelK3 (GETA),1,01 GOGO 3B SR2 (ERICA),Online
1,2,8,08 SO 2A SelK3 (GETA),2,02 SO 1C SR2 (QORIN),Offline
2,3,8,08 SO 2A SelK3 (GETA),3,03 SO 1B SR1 (TATIK),Offline
3,4,8,08 SO 2A SelK3 (GETA),4,04 SO 2A SR3 (TATIK),Offline
4,5,8,08 SO 2A SelK3 (GETA),5,05 GOGO 1B SelK3 (ERICA),Offline


✅ VERIFIKASI LULUS: Semua FK terhubung dengan benar.


In [30]:
# Cek nilai unik id_jadwal di detail sebelum mapping
print("Sebelum mapping (contoh):", df_jadwal_detail_insert['id_jadwal'].head(10).tolist())
# Lakukan mapping (jika belum)
df_jadwal_detail_insert['id_jadwal'] = df_jadwal_detail_insert['id_jadwal'].map(mapping_id_jadwal)
# Cek setelah mapping
print("Setelah mapping (contoh):", df_jadwal_detail_insert['id_jadwal'].head(10).tolist())

Sebelum mapping (contoh): [8, 8, 8, 8, 8, 8, 8, 8, 8, 8]
Setelah mapping (contoh): [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]


In [31]:
# Cek tipe data di old_id_list
print("Tipe data old_id_list pertama:", type(old_id_list[0]) if old_id_list else None)
print("Contoh 5 old_id_list:", old_id_list[:5])

# Cek tipe data idjadwal di detail
print("Tipe data idjadwal di detail:", df_detil_lama['idjadwal'].dtype)
print("Contoh 5 nilai idjadwal detail:", df_detil_lama['idjadwal'].head(5).tolist())

# Cek apakah key '8' ada di mapping (sebagai string dan integer)
print("Apakah '8' ada di mapping?", '8' in mapping_id_jadwal)
print("Apakah 8 ada di mapping?", 8 in mapping_id_jadwal)

Tipe data old_id_list pertama: <class 'str'>
Contoh 5 old_id_list: ['J000000023', 'J000000024', 'J000000025', 'J000000026', 'J000000027']
Tipe data idjadwal di detail: object
Contoh 5 nilai idjadwal detail: ['J000000030', 'J000000030', 'J000000030', 'J000000030', 'J000000030']
Apakah '8' ada di mapping? False
Apakah 8 ada di mapping? False
